# 화질 지표 계산

복원 결과가 "좋다" 는 것을 숫자로 말하려면 잣대가 필요하다. 잣대는 두 갈래다.

| | 지표 | 정답 영상이 | 재는 것 |
|---|---|---|---|
| **FR** 참조 있음 | PSNR, SSIM, ERGAS | **있어야 한다** | 정답과 얼마나 다른가 |
| **NR** 참조 없음 | NIQE, BRISQUE, PIQE | **필요 없다** | 이 영상 하나가 얼마나 자연스러운가 |

세 단계로 한다.

1. 데이터를 눈으로 본다
2. 같은 영상 세트에 여섯 지표를 모두 계산한다
3. 그림 옆에 점수를 붙여 본다

미리 뽑아둔 결과를 불러온다. **GPU 가 필요 없다.**

## 0. 준비

In [ ]:
import sys, json, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('sr_utils.py', 'iqa.py'):
    urllib.request.urlretrieve(f'{BASE}/lib/{f}', f)
    sys.modules.pop(f[:-3], None)
from sr_utils import *
import iqa
import pandas as pd

RES = f'{BASE}/results/comparison'
V = 'q2'          # 내려받은 파일 캐시 폴더. 결과가 바뀌면 이 값을 올린다
META = json.load(open(fetch(f'{RES}/meta.json', f'{V}/meta.json')))

# NIQE·BRISQUE 는 "정상 영상이란 이런 것" 이라는 기준이 있어야 점수가 나온다.
# 원 논문은 자연 사진으로 만들지만 위성 영상은 통계가 달라, 이 프로젝트의 HR 로
# 다시 잡은 것을 쓴다. -> 값은 이 데이터 안에서의 상대 비교로만 읽는다.
for f in ('niqe_model.npz', 'brisque_svr.npz'):
    fetch(f'{BASE}/results/iqa/{f}', f'{V}/iqa/{f}')
iqa.load_models(f'{V}/iqa')

VALS = sorted(k for k in META if k.startswith('val'))
TESTS_ = sorted(k for k in META if k.startswith('test'))
ORDER = ['Bicubic', 'SRCNN', 'VDSR', 'EDSR', 'SRGAN', 'ESRGAN', 'SwinIR', 'HAT']
CENTER, SIZE = (67, 370), 90      # 다른 페이지와 같은 확대 자리
CROP = (max(0, CENTER[0] - SIZE // 2),
        max(0, min(CENTER[1] - SIZE // 2, 384 - SIZE)), SIZE)


def load(scene, name):
    return imageio.imread(fetch(f'{RES}/{scene}/{name}.png', f'{V}/{scene}/{name}.png'))


print('정답 있는 패치:', ', '.join(VALS))
print('정답 없는 인천:', ', '.join(TESTS_))
print('지표:', ', '.join(iqa.ALL))

## 1. 데이터 시각화

먼저 무엇을 재는지 눈으로 본다.

**정답 HR 이 있는 패치 4장** — 여섯 지표를 모두 잴 수 있다.
**인천 2구역** — 실제 Sentinel-2 촬영본이라 정답이 없다. FR 세 가지는 계산 자체가 안 된다.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(12, 8.2))
for j, v in enumerate(VALS):
    a = ax[j // 3][j % 3]
    a.imshow(load(v, 'HR')); a.set_title(f'{v}   HR available', fontsize=11)
for j, t in enumerate(TESTS_):
    a = ax[1][j + 1]
    a.imshow(load(t, 'Bicubic'))
    a.set_title(f'{t}   no HR', fontsize=11, color='#c0504d')
for r in range(2):
    for c in range(3):
        ax[r][c].set_xticks([]); ax[r][c].set_yticks([])
plt.tight_layout(); plt.show()

한 패치를 골라 입력부터 정답까지 늘어놓는다. 아랫줄이 확대한 것이다.

In [ ]:
v = VALS[0]
zoom([('Original LR', load(v, 'LR')),
      ('Bicubic', load(v, 'Bicubic')),
      ('EDSR', load(v, 'EDSR')),
      ('ESRGAN', load(v, 'ESRGAN')),
      ('Target HR', load(v, 'HR'))],
     center=CENTER, size=SIZE, title=v)

## 2. 같은 영상 세트에 여섯 지표를 계산

`iqa.evaluate(결과, 정답)` 하나로 여섯 값이 나온다. 정답을 안 주면 NR 세 가지만 낸다.

검증 4장 × 여덟 결과에 모두 적용한다. 한 칸이 4장 평균이다.

In [ ]:
# 한 장에 대해 어떻게 나오는지 먼저 본다
one = iqa.evaluate(load(VALS[0], 'EDSR'), load(VALS[0], 'HR'))
for k, x in one.items():
    kind = 'FR' if k in iqa.FR else 'NR'
    arrow = '↑ 높을수록 좋다' if iqa.BETTER[k] == 'high' else '↓ 낮을수록 좋다'
    print(f'  [{kind}] {k.upper():8s} {x:9.3f}   {arrow}')

print('\n정답을 안 주면 NR 만 나온다:')
print(' ', iqa.evaluate(load(TESTS_[0], 'EDSR')))

In [ ]:
rows = []
for m in ORDER:
    acc = {k: [] for k in iqa.ALL}
    for v in VALS:
        for k, val in iqa.evaluate(load(v, m), load(v, 'HR')).items():
            acc[k].append(val)
    rows.append({'model': m, **{k: float(np.mean(a)) for k, a in acc.items()}})
df = pd.DataFrame(rows).set_index('model')

print('검증 4장 평균\n')
print(df.round(3).to_string())
print('\n지표마다 1등\n')
for k in iqa.ALL:
    win = df[k].idxmax() if iqa.BETTER[k] == 'high' else df[k].idxmin()
    arrow = '↑' if iqa.BETTER[k] == 'high' else '↓'
    print(f'  {k.upper():8s} {arrow}   {win:8s} ({df.loc[win, k]:.3f})')

## 3. 그림 옆에 점수를 붙여 본다

표만 보면 어느 숫자가 어느 그림인지 이어지지 않는다. 확대한 결과 바로 옆에 여섯 점수를
놓는다. **칸 색은 그 지표 안에서의 순위다 — 초록이 좋고 붉을수록 나쁘다.**

In [ ]:
v = VALS[0]
hr = load(v, 'HR')
items = [(m, load(v, m), iqa.evaluate(load(v, m), hr)) for m in ORDER]
iqa.report(items, crop=CROP, title=v)

**초록 칸이 한 줄로 모이는 모델이 없다.** 어떤 모델은 왼쪽 세 칸(FR)이 초록인데 오른쪽
세 칸(NR)이 붉고, 어떤 모델은 정반대다. 지표가 서로 다른 것을 재고 있다는 뜻이다.

다른 패치에서도 같은지 확인해 본다. `VALS[1]`, `VALS[2]` … 로 바꿔 보라.

In [ ]:
v = VALS[1]
hr = load(v, 'HR')
items = [(m, load(v, m), iqa.evaluate(load(v, m), hr)) for m in ORDER]
iqa.report(items, crop=CROP, title=v)

정답이 없는 인천에서도 해 본다. **FR 세 칸이 아예 없다.**

In [ ]:
t = TESTS_[0]
items = [(m, load(t, m), iqa.evaluate(load(t, m))) for m in ORDER]
iqa.report(items, metrics=iqa.NR, crop=(300, 300, 200), title=f'{t}  (no reference)')